In [7]:
# Install required dependencies (Colab)
#!pip -q install langchain langchain-community langchain-groq tavily-python faiss-cpu tiktoken python-dotenv


In [8]:
import os
from dotenv import load_dotenv

# 1) Try to load a local .env file if it exists in Colab runtime
load_dotenv()

def get_secret(name: str) -> str:
    """
    Read secrets from Colab (if available), fallback to environment.
    """
    # Colab secrets (recommended)
    try:
        from google.colab import userdata  # type: ignore
        value = userdata.get(name)
        if value:
            return value
    except Exception:
        pass

    # Environment variables fallback
    return os.getenv(name, "")

# Load required keys from Colab secrets/env
GROQ_API_KEY = get_secret("GROQ_API_KEY")
TAVILY_API_KEY = get_secret("TAVILY_API_KEY")

# Export them to env so downstream libs can read os.getenv(...)
if GROQ_API_KEY:
    os.environ["GROQ_API_KEY"] = GROQ_API_KEY
if TAVILY_API_KEY:
    os.environ["TAVILY_API_KEY"] = TAVILY_API_KEY

# Hard checks (now reliable)
assert os.getenv("GROQ_API_KEY"), "Missing GROQ_API_KEY (set it in Colab Secrets or create a .env in runtime)"
assert os.getenv("TAVILY_API_KEY"), "Missing TAVILY_API_KEY (set it in Colab Secrets or create a .env in runtime)"

print("✅ Keys loaded:")
print("  - GROQ_API_KEY:", "OK" if os.getenv("GROQ_API_KEY") else "MISSING")
print("  - TAVILY_API_KEY:", "OK" if os.getenv("TAVILY_API_KEY") else "MISSING")


✅ Keys loaded:
  - GROQ_API_KEY: OK
  - TAVILY_API_KEY: OK


In [9]:
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import FakeEmbeddings
from langchain_core.documents import Document

# Tiny demo corpus (keep it short and deterministic)
docs = [
    Document(
        page_content="Agentic RAG loops usually follow: reason -> retrieve -> read -> synthesize.",
        metadata={"source": "kb:agentic_rag_loop"},
    ),
    Document(
        page_content="Tools allow an agent to access external information like web search when internal context is not enough.",
        metadata={"source": "kb:tools"},
    ),
    Document(
        page_content="Source attribution is required: cite where the evidence came from.",
        metadata={"source": "kb:sources"},
    ),
]

embeddings = FakeEmbeddings(size=256)
vectorstore = FAISS.from_documents(docs, embeddings)
retriever = vectorstore.as_retriever(search_kwargs={"k": 2})

print("✅ Retriever ready with", len(docs), "documents")


✅ Retriever ready with 3 documents


In [10]:
from langchain_community.tools.tavily_search import TavilySearchResults

search_tool = TavilySearchResults(k=3)


/tmp/ipython-input-2172129817.py:3: LangChainDeprecationWarning: The class `TavilySearchResults` was deprecated in LangChain 0.3.25 and will be removed in 1.0. An updated version of the class exists in the `langchain-tavily package and should be used instead. To use it run `pip install -U `langchain-tavily` and import as `from `langchain_tavily import TavilySearch``.
  search_tool = TavilySearchResults(k=3)


In [11]:
from langchain_groq import ChatGroq

llm = ChatGroq(model="llama3-8b-8192")


In [12]:
def agentic_rag_answer(question: str) -> str:
    docs = retriever.invoke(question)
    context = "\n".join(d.page_content for d in docs)

    prompt = f"""
    Use the context to answer the question.
    Context:
    {context}

    Question:
    {question}

    Cite sources.
    """

    response = llm.invoke(prompt)
    return response.content
